# 1-vs-1 JSBSim skill-recognition dataset generator

This notebook creates a reproducible **100-flight** BVR-style dataset using the native JSBSim flight-dynamics engine and `rgcn_fusion.skill_transition.SkillManager`. It is intended as a small smoke-test corpus for skill recognition/forecasting—not as a validated combat model.

## Dataset contract

The revised planning contract is represented explicitly:

* one row per aircraft per timestamp, keyed by `(flight_id, timestamp_s, aircraft_id)`;
* 10 Hz observations (`sample_interval_s = 0.1`) while JSBSim integrates at 120 Hz;
* ownship state, controls, fuel, relative geometry and closure to the opponent;
* **ground-truth labels** `skill_label`, `skill_instance_id`, `skill_elapsed_s`, `skill_remaining_s`, `transition_reason`, and sampled skill parameters;
* `next_skill_label` and `time_to_next_skill_s` forecasting targets, derived only after a flight is complete;
* a flight-level manifest containing seeds, randomized initial conditions, outcome, and a train/validation/test split (split by flight to prevent trajectory leakage);
* one Tacview 2.x text ACMI file per flight.

Labels describe the controller currently commanding each aircraft. Features do not include label columns; select them with `FEATURE_COLUMNS` below. Skill transitions are stochastic but reproducible from the master seed. Randomized altitude, speed, heading, separation, aspect, lateral offset, fuel, and skill seed provide a range of starting configurations.

> **Operational note:** generated engagements are synthetic and unclassified. The simple launch/threat logic is deliberately abstract and must not be interpreted as weapon-performance data.


## Environment

Run from the repository root. JSBSim's Python package wraps the native C++ FDM required by `SkillManager`. Uncomment the install line in a fresh notebook environment.

In [ ]:
# %pip install "jsbsim>=1.2.0" "pandas>=2.0" "pyarrow>=14"
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import UTC, datetime, timedelta
from pathlib import Path
import json, math, random

import pandas as pd
import jsbsim

from rgcn_fusion.skill_transition import (
    FlightControls, NativeJSBSimAdapter, SkillManager, default_skill_specs,
)

REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "generated" / "bvr_skill_dataset"
ACMI_DIR = OUTPUT_DIR / "tacview"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ACMI_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 20260911
N_FLIGHTS = 100
DURATION_S = 60.0
SAMPLE_INTERVAL_S = 0.1       # contract: never coarser than 10 Hz
INTEGRATION_DT_S = 1.0 / 120  # native FDM integration rate
AIRCRAFT_MODEL = "f16"
assert SAMPLE_INTERVAL_S <= 0.1


## Configuration sampling and geometry

The sampler uses bounded distributions and opposing headings with randomized aspect. Each flight records the exact sampled values in the manifest, allowing complete replay.

In [ ]:
@dataclass(frozen=True)
class InitialCondition:
    altitude_ft: float
    speed_kts: float
    heading_blue_deg: float
    heading_red_deg: float
    separation_nm: float
    lateral_offset_nm: float
    fuel_fraction_blue: float
    fuel_fraction_red: float


def sample_initial_condition(rng: random.Random) -> InitialCondition:
    blue_heading = rng.uniform(0.0, 360.0)
    return InitialCondition(
        altitude_ft=rng.uniform(20_000.0, 40_000.0),
        speed_kts=rng.uniform(350.0, 520.0),
        heading_blue_deg=blue_heading,
        heading_red_deg=(blue_heading + 180.0 + rng.uniform(-45.0, 45.0)) % 360.0,
        separation_nm=rng.uniform(25.0, 65.0),
        lateral_offset_nm=rng.uniform(-12.0, 12.0),
        fuel_fraction_blue=rng.uniform(0.55, 1.0),
        fuel_fraction_red=rng.uniform(0.55, 1.0),
    )


def haversine_nm(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2-p1, math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 3440.065 * 2 * math.atan2(math.sqrt(a), math.sqrt(max(0.0, 1-a)))


def bearing_deg(lat1, lon1, lat2, lon2):
    p1, p2, dl = math.radians(lat1), math.radians(lat2), math.radians(lon2-lon1)
    return math.degrees(math.atan2(math.sin(dl)*math.cos(p2),
        math.cos(p1)*math.sin(p2)-math.sin(p1)*math.cos(p2)*math.cos(dl))) % 360.0


def angle_error_deg(target, current):
    return (target-current+180.0) % 360.0-180.0


## Native JSBSim setup and skill controllers

Controllers are intentionally simple, but every label corresponds exactly to the `SkillManager` state that produced the controls. The tactical observer adds symmetric relative geometry and abstract launch/threat flags to the physical FDM state.

In [ ]:
def make_fdm(lat_deg, lon_deg, ic: InitialCondition, heading_deg, fuel_fraction):
    fdm = jsbsim.FGFDMExec(None)
    fdm.set_debug_level(0)
    fdm.load_model(AIRCRAFT_MODEL)
    fdm.set_dt(INTEGRATION_DT_S)
    fdm["ic/lat-geod-deg"] = lat_deg
    fdm["ic/long-gc-deg"] = lon_deg
    fdm["ic/h-sl-ft"] = ic.altitude_ft
    fdm["ic/vc-kts"] = ic.speed_kts
    fdm["ic/psi-true-deg"] = heading_deg
    fdm["ic/gamma-deg"] = 0.0
    # Most bundled models expose these tank contents; skip absent properties.
    for name in ("propulsion/tank[0]/contents-lbs", "propulsion/tank[1]/contents-lbs"):
        try:
            capacity = float(fdm.get_property_value(name.replace("contents", "capacity")))
            if capacity > 0: fdm[name] = capacity * fuel_fraction
        except Exception:
            pass
    if not fdm.run_ic():
        raise RuntimeError(f"JSBSim failed to initialise {AIRCRAFT_MODEL}")
    return fdm


def controller(skill):
    def command(obs, p):
        err = angle_error_deg(obs.get("target_bearing_deg", obs["heading_deg"]), obs["heading_deg"])
        toward = max(-1.0, min(1.0, err / 45.0))
        bank_noise = p.get("bank_command", 0.0) * 0.15
        if skill in {"pursue_target", "recommit", "launch", "support_missile"}: bank = toward + bank_noise
        elif skill == "crank_maneuver": bank = (0.65 if err >= 0 else -0.65) + bank_noise
        elif skill in {"turn_cold", "disengage", "missile_evasion"}: bank = -toward + (0.55 if err < 0 else -0.55)
        elif skill == "search": bank = 0.3 + bank_noise
        else: bank = bank_noise
        elevator = max(-0.25, min(0.25, (30_000.0-obs["altitude_ft"]) / 20_000.0))
        throttle = 1.0 if skill in {"pursue_target", "recommit", "missile_evasion"} else 0.82
        return FlightControls(max(-1, min(1, bank)), elevator, 0.0, throttle)
    return command

SPECS = default_skill_specs()
CONTROLLERS = {name: controller(name) for name in SPECS}


In [ ]:
def physical_state(fdm):
    get = fdm.get_property_value
    return {
        "latitude_deg": get("position/lat-geod-deg"),
        "longitude_deg": get("position/long-gc-deg"),
        "altitude_ft": get("position/h-sl-ft"),
        "heading_deg": get("attitude/psi-deg") % 360.0,
        "pitch_deg": get("attitude/theta-deg"),
        "roll_deg": get("attitude/phi-deg"),
        "airspeed_kts": get("velocities/vc-kts"),
        "vertical_speed_fps": get("velocities/h-dot-fps"),
        "north_velocity_fps": get("velocities/v-north-fps"),
        "east_velocity_fps": get("velocities/v-east-fps"),
        "fuel_lbs": get("propulsion/total-fuel-lbs"),
    }


def tactical_observations(blue, red, previous_range, dt):
    bs, rs = physical_state(blue), physical_state(red)
    distance = haversine_nm(bs["latitude_deg"], bs["longitude_deg"], rs["latitude_deg"], rs["longitude_deg"])
    closure = 0.0 if previous_range is None else (previous_range-distance) * 3600.0 / dt
    observations = []
    for own, other, sign in ((bs, rs, 1.0), (rs, bs, -1.0)):
        brg = bearing_deg(own["latitude_deg"], own["longitude_deg"], other["latitude_deg"], other["longitude_deg"])
        observations.append({**own,
            "target_bearing_deg": brg,
            "target_aspect_deg": angle_error_deg((brg+180)%360, other["heading_deg"]),
            "bearing_error_deg": angle_error_deg(brg, own["heading_deg"]),
            "range_nm": distance, "closure_kts": closure,
            "relative_altitude_ft": other["altitude_ft"]-own["altitude_ft"],
            "target_in_launch_envelope": distance <= 35.0 and abs(angle_error_deg(brg, own["heading_deg"])) < 50.0,
            "incoming_active_missile": distance <= 18.0,
            "fuel_fraction": max(0.0, min(1.0, own["fuel_lbs"] / 7000.0)),
            "bingo_fuel_fraction": 0.15, "weapons_remaining": 2,
            "target_destroyed": False})
    return observations, distance


## Rollout, exact 10 Hz capture, labels, and Tacview export

JSBSim takes 12 native steps between samples. Rows are captured at integer sample indices to avoid floating-point drift. Tacview uses longitude/latitude/altitude (metres), followed by roll/pitch/yaw, and includes the skill as an event so timelines can be inspected visually.

In [ ]:
FEATURE_COLUMNS = [
    "altitude_ft", "airspeed_kts", "heading_deg", "pitch_deg", "roll_deg",
    "vertical_speed_fps", "north_velocity_fps", "east_velocity_fps", "fuel_fraction",
    "range_nm", "closure_kts", "target_bearing_deg", "bearing_error_deg",
    "target_aspect_deg", "relative_altitude_ft", "aileron_cmd", "elevator_cmd", "throttle_cmd",
]
LABEL_COLUMNS = ["skill_label", "skill_instance_id", "skill_elapsed_s", "skill_remaining_s",
                 "transition_reason", "skill_parameters_json", "next_skill_label", "time_to_next_skill_s"]


def add_forecast_targets(rows):
    by_aircraft = {}
    for row in rows: by_aircraft.setdefault(row["aircraft_id"], []).append(row)
    for trajectory in by_aircraft.values():
        next_change = None
        for i in range(len(trajectory)-1, -1, -1):
            if i < len(trajectory)-1 and trajectory[i]["skill_label"] != trajectory[i+1]["skill_label"]:
                next_change = trajectory[i+1]
            trajectory[i]["next_skill_label"] = None if next_change is None else next_change["skill_label"]
            trajectory[i]["time_to_next_skill_s"] = None if next_change is None else round(next_change["timestamp_s"]-trajectory[i]["timestamp_s"], 6)


def write_acmi(flight_id, rows, reference_time):
    path = ACMI_DIR / f"{flight_id}.acmi"
    colors = {"blue": "Blue", "red": "Red"}; ids = {"blue": "100", "red": "200"}
    lines = ["FileType=text/acmi/tacview", "FileVersion=2.2",
             f"0,ReferenceTime={reference_time.isoformat().replace('+00:00','Z')}",
             f"0,Title={flight_id},DataRecorder=JSBSim SkillManager notebook"]
    last_skill = {}
    for row in rows:
        aid, obj = row["aircraft_id"], ids[row["aircraft_id"]]
        if row["sample_index"] == 0:
            lines.append(f"{obj},T=,Name={aid},Type=Air+FixedWing,Coalition={aid},Color={colors[aid]}")
        lines.append(f"#{row['timestamp_s']:.1f}")
        alt_m = row["altitude_ft"] * 0.3048
        lines.append(f"{obj},T={row['longitude_deg']:.8f}|{row['latitude_deg']:.8f}|{alt_m:.2f}|{row['roll_deg']:.3f}|{row['pitch_deg']:.3f}|{row['heading_deg']:.3f}")
        if last_skill.get(aid) != row["skill_label"]:
            lines.append(f"0,Event=Message|{obj}|skill={row['skill_label']}; reason={row['transition_reason']}")
            last_skill[aid] = row["skill_label"]
    path.write_text("\n".join(lines)+"\n", encoding="utf-8")
    return path


In [ ]:
def run_flight(flight_index, seed, duration_s=DURATION_S):
    rng = random.Random(seed); ic = sample_initial_condition(rng)
    # Offset Red from Blue around a neutral reference point.
    lat0, lon0 = 36.0 + rng.uniform(-2, 2), -115.0 + rng.uniform(-2, 2)
    theta = math.radians(ic.heading_blue_deg)
    north_nm = ic.separation_nm*math.cos(theta) - ic.lateral_offset_nm*math.sin(theta)
    east_nm = ic.separation_nm*math.sin(theta) + ic.lateral_offset_nm*math.cos(theta)
    red_lat, red_lon = lat0+north_nm/60.0, lon0+east_nm/(60.0*math.cos(math.radians(lat0)))
    blue = make_fdm(lat0, lon0, ic, ic.heading_blue_deg, ic.fuel_fraction_blue)
    red = make_fdm(red_lat, red_lon, ic, ic.heading_red_deg, ic.fuel_fraction_red)
    managers = [SkillManager(SPECS, CONTROLLERS, seed=seed*2),
                SkillManager(SPECS, CONTROLLERS, seed=seed*2+1)]
    adapters = [NativeJSBSimAdapter(blue, physical_state), NativeJSBSimAdapter(red, physical_state)]
    flight_id = f"flight_{flight_index:03d}"; rows=[]; previous_range=None
    integration_steps = round(SAMPLE_INTERVAL_S/INTEGRATION_DT_S)
    assert math.isclose(integration_steps*INTEGRATION_DT_S, SAMPLE_INTERVAL_S)
    n_samples = math.floor(duration_s/SAMPLE_INTERVAL_S)+1
    instance = [-1, -1]; previous_state = [None, None]
    for sample_index in range(n_samples):
        now = sample_index*SAMPLE_INTERVAL_S
        observations, current_range = tactical_observations(blue, red, previous_range, SAMPLE_INTERVAL_S)
        controls=[]
        for side, (manager, obs) in enumerate(zip(managers, observations)):
            control = manager.controls(obs, now); controls.append(control); state=manager.state
            if state is not previous_state[side]: instance[side]+=1; previous_state[side]=state
            rows.append({"flight_id":flight_id, "sample_index":sample_index,
                "timestamp_s":round(now, 6), "aircraft_id":("blue","red")[side],
                **{k: obs[k] for k in FEATURE_COLUMNS if k in obs},
                "latitude_deg":obs["latitude_deg"], "longitude_deg":obs["longitude_deg"],
                "fuel_fraction":obs["fuel_fraction"], "aileron_cmd":control.aileron,
                "elevator_cmd":control.elevator, "throttle_cmd":control.throttle,
                "skill_label":state.skill, "skill_instance_id":instance[side],
                "skill_elapsed_s":round(now-state.entered_at, 6),
                "skill_remaining_s":round(max(0.0,state.expires_at-now), 6),
                "transition_reason":state.transition_reason,
                "skill_parameters_json":json.dumps(dict(state.parameters), sort_keys=True)})
        if sample_index < n_samples-1:
            for _ in range(integration_steps):
                adapters[0].step(controls[0], INTEGRATION_DT_S)
                adapters[1].step(controls[1], INTEGRATION_DT_S)
        previous_range=current_range
    add_forecast_targets(rows)
    reference=datetime(2026,1,1,tzinfo=UTC)+timedelta(hours=flight_index)
    acmi=write_acmi(flight_id, rows, reference)
    return rows, {"flight_id":flight_id, "seed":seed, "aircraft_model":AIRCRAFT_MODEL,
        "duration_s":duration_s, "sample_interval_s":SAMPLE_INTERVAL_S,
        "integration_dt_s":INTEGRATION_DT_S, "initial_condition":asdict(ic),
        "reference_time_utc":reference.isoformat(), "tacview_file":str(acmi.relative_to(OUTPUT_DIR)),
        "outcome":"timeout"}


## Generate the corpus

The default cell runs approximately 100 × 60 seconds of two-aircraft dynamics. Reduce `N_FLIGHTS` while developing. CSV is always written; Parquet gives typed, compact training input and requires `pyarrow`.

In [ ]:
all_rows=[]; manifest=[]
for flight_index in range(N_FLIGHTS):
    seed=MASTER_SEED+flight_index
    rows, metadata=run_flight(flight_index, seed)
    # Stable flight-level partition: no samples from one engagement cross a split.
    bucket=random.Random(seed ^ 0x5EED).random()
    metadata["split"] = "train" if bucket < .70 else ("validation" if bucket < .85 else "test")
    all_rows.extend(rows); manifest.append(metadata)

dataset=pd.DataFrame(all_rows)
dataset.to_csv(OUTPUT_DIR/"samples.csv", index=False)
dataset.to_parquet(OUTPUT_DIR/"samples.parquet", index=False)
(OUTPUT_DIR/"manifest.json").write_text(json.dumps({
    "schema_version":"bvr-skill-recognition-1.0", "master_seed":MASTER_SEED,
    "feature_columns":FEATURE_COLUMNS, "label_columns":LABEL_COLUMNS, "flights":manifest
}, indent=2)+"\n", encoding="utf-8")
print(f"Wrote {len(dataset):,} rows, {len(manifest)} flights and {len(list(ACMI_DIR.glob('*.acmi')))} ACMI files")
dataset.head()


## Validation gates

These assertions enforce the sampling, relational symmetry, label, forecast, configuration-diversity, file-count, and leakage-prevention requirements before training.

In [ ]:
assert dataset["flight_id"].nunique() == N_FLIGHTS
assert not dataset.duplicated(["flight_id","timestamp_s","aircraft_id"]).any()
counts=dataset.groupby(["flight_id","timestamp_s"])["aircraft_id"].nunique()
assert counts.eq(2).all()
intervals=dataset.sort_values("timestamp_s").groupby(["flight_id","aircraft_id"])["timestamp_s"].diff().dropna()
assert intervals.le(0.1+1e-9).all() and intervals.gt(0).all()
assert dataset["skill_label"].isin(SPECS).all()
assert dataset[FEATURE_COLUMNS].notna().all().all()
assert (dataset["skill_elapsed_s"] >= 0).all() and (dataset["skill_remaining_s"] >= 0).all()
assert len(list(ACMI_DIR.glob("*.acmi"))) == N_FLIGHTS
assert len({m["seed"] for m in manifest}) == N_FLIGHTS
assert len({round(m["initial_condition"]["altitude_ft"],1) for m in manifest}) > 1
split_by_flight={m["flight_id"]:m["split"] for m in manifest}
assert len(split_by_flight) == N_FLIGHTS
print(dataset.groupby("skill_label").size().sort_values(ascending=False))
print(pd.Series(split_by_flight).value_counts())


## Training use

Use `FEATURE_COLUMNS` for model inputs and one of the label targets for supervision. Keep the manifest's flight-level splits intact. In particular, do **not** feed `skill_label`, skill timing, transition reason, sampled parameters, or future-label columns into a recognizer: those are ground truth and would leak the answer. Fit scalers on the training flights only. Sequence windows should never cross `flight_id` or `aircraft_id` boundaries.
